In [ ]:
# Step 1: Install Required Dependencies
%pip install --upgrade pip

# Uninstall conflicting packages
%pip uninstall -y langchain_classic langchain-core langchain-openai langchain-community langchain langchain-chroma chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25 weaviate-client ragas wikipedia langchain-weaviate langchain-together langchain-experimental tiktoken langgraph langchain-tavily

%pip install langchain==1.1.0
%pip install langgraph==1.0.4
%pip install langchain-openai==1.1.0
%pip install langchain-chroma==1.0.0
%pip install chromadb==1.3.5
%pip install python-dotenv==1.2.1

In [ ]:
# Step 2: Import Core Libraries and Configure Environment
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langsmith import Client
from langchain_core.output_parsers import StrOutputParser
from langchain_chroma import Chroma
import chromadb
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END

_ = load_dotenv(dotenv_path='env.txt')
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings()

In [ ]:
# Step 3: Build the Basic Agent
# Define Agent State with memory placeholders
class AgentState(TypedDict):
    """State container for agent memory and messages"""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    working_memory: dict  # Short-term context
    episodic_recall: list  # Retrieved past experiences
    semantic_facts: dict  # Retrieved knowledge

# Initialize vector store for future memory storage
vector_store = Chroma(
    collection_name="agent_memory",
    embedding_function=embeddings,
    persist_directory="./memory_store"
)

# Create base prompt and chain
base_prompt = PromptTemplate.from_template("""
You are a helpful assistant with memory capabilities.

Current conversation:
{messages}

Please respond to the latest message.
""")

output_parser = StrOutputParser()

# Define agent node
def agent_node(state: AgentState) -> dict:
    """Core agent logic - processes messages and generates responses"""
    messages = state["messages"][-5:] if state["messages"] else []
    formatted_messages = "\n".join([
        f"{msg.type}: {msg.content}" 
        for msg in messages
    ])
    
    chain = base_prompt | llm | output_parser
    response = chain.invoke({"messages": formatted_messages})
    
    return {"messages": [("assistant", response)]}

# Build and compile the graph
workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.set_entry_point("agent")
workflow.add_edge("agent", END)

app = workflow.compile()

In [ ]:
# Step 4: Test the Basic Agent
# Test the agent
test_input = {
    "messages": [("user", "Hello! What's the capital of France?")]
}

result = app.invoke(test_input)
print(result["messages"][-1].content)